# Twin-2K-500-Mega-Study: Manual Verification & Kev Long-Format Export

[`LLM-Digital-Twin/Twin-2K-500-Mega-Study`](https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500-Mega-Study) extends [`LLM-Digital-Twin/Twin-2K-500`](../Twin-2K-500) by re-surveying **1,788** respondents from the exact same `Twin-2K-500` Prolific panel (`PID` `"pid_1"`..`"pid_2058"`) across **19** follow-up behavioral studies (**13,816** respondent-study completions).

Unlike `Twin-2K-500`, this repository ships raw per-respondent Qualtrics survey flow JSONs (`survey_json_with_human_response`) across 19 split Parquet files without a pre-built question catalog or flat CSV response matrix, and omits individual-level demographics in 18 of 19 splits because respondent personas are linked via `PID` to `Twin-2K-500` Waves 1–3.

This notebook mirrors [`../Twin-2K-500/data.ipynb`](../Twin-2K-500/data.ipynb):
1. **Raw splits & recursive survey flow catalog** — downloads/verifies all 19 split Parquets and builds the canonical **1,748-entry** `(study, QuestionID)` catalog across nested `Block`, `Branch`, `Group`, and `Randomizer` flow elements.
2. **Respondent `PID` linkage & personas** — verifies 100% `PID` membership in `Twin-2K-500` Waves 1–3 (`CASEID = "twin_<pid>"`) and bit-for-bit persona alignment with `twin2k_US_long.parquet`.
3. **Manual verification audit (`question_audit.csv`) & Kev preview helper** — validates 1-to-1 coverage of all 1,748 catalog entries, inspects flagged items, and defines the `KevEncoder` preview and fail-fast long-format extractor.
4. **Family-by-family inspection (`1_Matrix_Bipolar`, `2_Matrix_Likert`, `3_MC_IdeaEval`, `4_MC_Single`, and excluded families)** — extracts each discrete family and renders full Kev prompt/target previews.
5. **Strict `KevEncoder` validation & Parquet export** — verifies `KevEncoder.prepare(strict=True)` on every unique question and exports `twin2k_megastudy_US_long.parquet` matching the exact 24-column schema of `twin2k_US_long.parquet`.

## 1. Raw Mega-Study splits and survey flow catalog

In [1]:
import html
import json
import re
import sys
import textwrap
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from IPython.display import display

REPO_ID = "LLM-Digital-Twin/Twin-2K-500-Mega-Study"
MEGA_DIR = Path.cwd() if Path.cwd().name == "Twin-2K-500-Mega-Study" else Path("code/data/Twin-2K-500-Mega-Study").resolve()
HF_DIR = MEGA_DIR / "hf"
RAW_DATA_DIR = HF_DIR / "data"
TWIN_DIR = MEGA_DIR.parent / "Twin-2K-500"
CODE_DIR = MEGA_DIR.parents[1]

for _p in (str(CODE_DIR), str(CODE_DIR / "kev")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

if not RAW_DATA_DIR.exists() or len(list(RAW_DATA_DIR.glob("*.parquet"))) != 19:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=HF_DIR,
        allow_patterns=["data/*.parquet", "README.md"],
    )

parquet_files = sorted(RAW_DATA_DIR.glob("*.parquet"))
if len(parquet_files) != 19:
    raise ValueError(f"Expected 19 split parquet files in {RAW_DATA_DIR}, found {len(parquet_files)}")
print(f"Verified {len(parquet_files)} split parquet files in {RAW_DATA_DIR}")

Verified 19 split parquet files in /home/dw/github/decision-models-finetuning/code/data/Twin-2K-500-Mega-Study/hf/data


In [2]:
ALLOWED_TOP_KEYS = ("Elements", "EmbeddedData", "Metadata")
ALLOWED_ELEMENT_TYPES = {"Block", "Branch", "Group", "Randomizer"}
ALLOWED_BLOCK_KEYS = ("BlockID", "BlockName", "BlockType", "ElementType", "Questions")
EXPECTED_PARQUET_COLS = ["PID", "survey_json_with_human_response", "survey_text"]
BR_RE = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)


def norm_str(text: str) -> str:
    return " ".join(BR_RE.sub(" ", html.unescape(text)).split())


def clean_text(study: str, qid: str, text: str) -> str:
    norm = norm_str(text)
    if study == "idea_evaluation" and re.match(r"^\d+_QID", qid):
        norm = re.sub(r"Idea #\d+ out of \d+:", "Idea:", norm)
    return norm


def iter_blocks(elems, depth=0):
    for e in elems:
        et = e["ElementType"]
        if et not in ALLOWED_ELEMENT_TYPES:
            raise ValueError(f"Unexpected ElementType: {et!r}")
        if et == "Block":
            if tuple(sorted(e.keys())) != ALLOWED_BLOCK_KEYS:
                raise ValueError(f"Unexpected Block keys: {sorted(e.keys())}")
            yield e, depth
        else:
            yield from iter_blocks(e["Elements"], depth + 1)


def has_answer(qt: str, sel: str, ans) -> bool:
    if qt == "DB":
        return False
    if not ans:
        return False
    if qt == "MC" and sel in ("SAVR", "SAHR", "DL"):
        return ans["SelectedByPosition"] is not None
    if qt == "MC" and sel == "MAVR":
        return bool(ans["SelectedByPosition"])
    if qt == "Matrix":
        return any(x is not None for x in ans["SelectedByPosition"])
    if qt in ("Slider", "CS"):
        return any(x is not None for x in ans["Values"])
    if qt == "TE":
        t = ans["Text"]
        if isinstance(t, list):
            return any(bool(str(x).strip()) for x in t if x is not None)
        return t is not None and bool(str(t).strip())
    raise ValueError(f"Unexpected QuestionType/Selector: {qt}:{sel}")


study_dfs = {}
study_summary_rows = []
catalog_entries = []

for p in parquet_files:
    study = p.name.removesuffix("-00000-of-00001.parquet")
    df = pd.read_parquet(p)
    if list(df.columns) != EXPECTED_PARQUET_COLS:
        raise ValueError(f"{study}: unexpected columns {list(df.columns)}")
    if df["PID"].isna().any() or df["PID"].duplicated().any():
        raise ValueError(f"{study}: missing or duplicate PIDs")
    study_dfs[study] = df[["PID", "survey_json_with_human_response"]].copy()

    slots = OrderedDict()
    nested_blocks_seen = 0
    for raw in df["survey_json_with_human_response"]:
        obj = json.loads(raw)
        if tuple(sorted(obj.keys())) != ALLOWED_TOP_KEYS:
            raise ValueError(f"{study}: unexpected top-level JSON keys {sorted(obj.keys())}")
        seen_qids_in_resp = set()
        for b, depth in iter_blocks(obj["Elements"]):
            if depth > 0:
                nested_blocks_seen += 1
            bname = b["BlockName"]
            for q in b["Questions"]:
                qid = q["QuestionID"]
                if qid in seen_qids_in_resp:
                    raise ValueError(f"{study}: duplicate QuestionID {qid!r} within a single respondent")
                seen_qids_in_resp.add(qid)
                qt = q["QuestionType"]
                sel = q["Settings"]["Selector"] if "Settings" in q else ""
                if qid not in slots:
                    rows = [norm_str(r) for r in q["Rows"]] if "Rows" in q else []
                    if qt == "Matrix":
                        opts = [norm_str(c) for c in q["Columns"]]
                        n_sub = len(rows)
                    elif qt == "MC":
                        opts = [norm_str(o) for o in q["Options"]]
                        n_sub = len(opts) if sel == "MAVR" else 1
                    elif qt == "DB":
                        opts, n_sub = [], 0
                    elif qt == "TE" and sel == "FORM":
                        opts, n_sub = [], len(rows)
                    elif qt in ("Slider", "CS"):
                        stmts = q["Statements"] if "Statements" in q else rows
                        opts, n_sub = [], max(1, len(stmts))
                    else:
                        opts, n_sub = [], 1
                    slots[qid] = {
                        "study": study,
                        "QuestionID": qid,
                        "BlockName": bname,
                        "QuestionType": qt,
                        "Selector": sel,
                        "n_subitems": n_sub,
                        "n_seen": 0,
                        "n_answered": 0,
                        "QuestionText": clean_text(study, qid, q["QuestionText"]),
                        "Rows": rows,
                        "Options_or_Columns": opts,
                    }
                else:
                    if (slots[qid]["QuestionType"], slots[qid]["Selector"]) != (qt, sel):
                        raise ValueError(f"{study}:{qid} changed QuestionType/Selector across respondents")
                slots[qid]["n_seen"] += 1
                if has_answer(qt, sel, q["Answers"] if qt != "DB" else None):
                    slots[qid]["n_answered"] += 1

    slot_order = {k: i for i, k in enumerate(slots.keys())}

    def sort_key(item):
        qid, _ = item
        if study == "idea_evaluation" and re.match(r"^\d+_QID", qid):
            num_str, base_q = qid.split("_", 1)
            arm_order = {"QID7": 1, "QID13": 2, "QID14": 3, "QID23": 4, "QID24": 5, "QID25": 6}[base_q]
            return (1, arm_order, int(num_str))
        return (0, 0, slot_order[qid])

    ordered_slots = [st for _, st in sorted(slots.items(), key=sort_key)]
    catalog_entries.extend(ordered_slots)

    qt_counts = pd.Series([s["QuestionType"] for s in ordered_slots]).value_counts()
    study_summary_rows.append({
        "study": study,
        "n_respondents": len(df),
        "n_slots": len(ordered_slots),
        "nested_block_instances": nested_blocks_seen,
        "MC": int(qt_counts.get("MC", 0)),
        "Matrix": int(qt_counts.get("Matrix", 0)),
        "DB": int(qt_counts.get("DB", 0)),
        "TE": int(qt_counts.get("TE", 0)),
        "Slider": int(qt_counts.get("Slider", 0)),
        "CS": int(qt_counts.get("CS", 0)),
    })

catalog_df = pd.DataFrame(catalog_entries)
study_summary_df = pd.DataFrame(study_summary_rows)
print(
    f"Total respondent-study rows: {study_summary_df['n_respondents'].sum():,} across {len(study_summary_df)} studies | "
    f"Canonical catalog entries: {len(catalog_df):,}"
)
display(study_summary_df)

Total respondent-study rows: 13,816 across 19 studies | Canonical catalog entries: 1,748


,study,n_respondents,n_slots,nested_block_instances,MC,Matrix,DB,TE,Slider,CS
0,accuracy_nudges,1003,69,18047,28,1,31,1,8,0
1,affective_priming,1000,15,4000,1,6,5,3,0,0
2,consumer_minimalism,200,7,1200,4,1,2,0,0,0
3,context_effects,501,15,1503,10,0,5,0,0,0
4,default_eric,600,9,1800,4,0,5,0,0,0
5,digital_certification,600,14,1200,5,1,4,4,0,0
6,hiring_algorithms,999,26,999,18,8,0,0,0,0
7,idea_evaluation,1174,1212,2348,1204,0,8,0,0,0
8,idea_generation,200,22,1200,0,0,9,13,0,0
9,infotainment,300,30,0,21,3,5,1,0,0


In [3]:
# Breakdown of all 1,748 catalog entries by (QuestionType, Selector)
display(
    catalog_df.groupby(["QuestionType", "Selector"], as_index=False)
    .agg(
        n_studies=("study", "nunique"),
        n_slots=("QuestionID", "count"),
        n_subitems=("n_subitems", "sum"),
        total_answered=("n_answered", "sum"),
    )
    .sort_values(["QuestionType", "Selector"], ignore_index=True)
)

,QuestionType,Selector,n_studies,n_slots,n_subitems,total_answered
0,CS,VRTL,1,8,40,2008
1,DB,,18,177,0,0
2,MC,DL,1,4,4,4800
3,MC,MAVR,2,18,141,10115
4,MC,SAHR,11,157,157,59998
5,MC,SAVR,13,1282,1282,98232
6,Matrix,Bipolar,3,12,12,9293
7,Matrix,Likert,8,31,202,16089
8,Slider,HSLIDER,2,10,10,10026
9,TE,ESTB,2,5,5,2439


## 2. Respondent linkage and Twin-2K-500 Wave 1-3 personas

In [4]:
from data import persona

w13_csv_path = TWIN_DIR / "hf" / "question_catalog_and_human_response_csv" / "wave1_3_response_label.csv"
if not w13_csv_path.exists():
    raise FileNotFoundError(f"Twin-2K-500 Wave 1-3 CSV not found: {w13_csv_path}")

w13_df = pd.read_csv(w13_csv_path, dtype=str)
w13_df["pid"] = w13_df["pid"].astype(int)
w13_df = w13_df.set_index("pid").sort_index()

# Verify 100% PID linkage across all 19 Mega-Study splits
pid_study_counts = {}
for study, df in study_dfs.items():
    for pid_str in df["PID"]:
        if not pid_str.startswith("pid_"):
            raise ValueError(f"{study}: unexpected PID string {pid_str!r}")
        pid_int = int(pid_str.removeprefix("pid_"))
        if pid_int not in w13_df.index:
            raise KeyError(f"{study}: {pid_str!r} missing from Twin-2K-500 Wave 1-3")
        pid_study_counts[pid_int] = pid_study_counts.get(pid_int, 0) + 1

mega_pids = sorted(pid_study_counts.keys())
studies_per_resp = pd.Series(pid_study_counts).sort_index()

# Map Wave 1-3 demographics via code/data/persona.py and verify exact match with twin2k_US_long.parquet
twin_mapped = persona.apply_persona_mapping(w13_df, dataset="twin2k")
PERSONA_COLS = persona.DATASET_PERSONA_COLS["twin2k"]
persona_base_df = twin_mapped[PERSONA_COLS].copy()
persona_base_df.insert(0, "CASEID", [f"twin_{pid}" for pid in persona_base_df.index])

twin_parquet_path = TWIN_DIR / "twin2k_US_long.parquet"
if twin_parquet_path.exists():
    twin_ref_personas = (
        pd.read_parquet(twin_parquet_path, columns=["CASEID"] + PERSONA_COLS)
        .drop_duplicates("CASEID")
        .set_index("CASEID")
    )
    chk_personas = persona_base_df.set_index("CASEID").loc[twin_ref_personas.index]
    for c in PERSONA_COLS:
        s1, s2 = chk_personas[c], twin_ref_personas[c]
        if ((s1 != s2) & ~(s1.isna() & s2.isna())).any():
            raise ValueError(f"Persona value mismatch vs twin2k_US_long.parquet on {c}")
    print(f"Verified 100% persona value identity with {twin_parquet_path.name} across all {len(w13_df):,} respondents.")

print(
    f"Mega-Study unique respondents: {len(mega_pids):,} / {len(w13_df):,} Twin-2K-500 panelists ({len(mega_pids)/len(w13_df):.1%})\n"
    f"Studies completed per Mega-Study respondent: min={studies_per_resp.min()}, "
    f"median={studies_per_resp.median():.0f}, mean={studies_per_resp.mean():.2f}, max={studies_per_resp.max()}"
)
display(persona_base_df.loc[mega_pids[:5]])

Verified 100% persona value identity with twin2k_US_long.parquet across all 2,058 respondents.
Mega-Study unique respondents: 1,788 / 2,058 Twin-2K-500 panelists (86.9%)
Studies completed per Mega-Study respondent: min=1, median=8, mean=7.73, max=18


,CASEID,c_alphan,US_REG,SEX,AGE,EDULEVEL,RACE,MARITAL,RELIGGRP,ATTEND,US_PRTY,US_INC,PARTY_LR,HHSIZE,MAINSTAT
pid,,,,,,,,,,,,,,,
1,twin_1,United States,Midwest,Female,30-49,High school or some college,Black,Married,Nothing in particular,A few times a year,Independent,"$50,000 - $75,000",Moderate,3,Part-time employment
2,twin_2,United States,Midwest,Female,50-64,High school or some college,White,Divorced,Nothing in particular,Never,Democrat,"$30,000 - $50,000",Very liberal,2,Retired
3,twin_3,United States,South,Male,65+,High school or some college,White,Married,Protestant,Seldom,Independent,"$30,000 - $50,000",Moderate,2,Retired
4,twin_4,United States,South,Male,50-64,College graduate/some postgrad,White,Married,Other,Never,Democrat,"$75,000 - $100,000",Very liberal,2,Full-time employment
5,twin_5,United States,South,Female,30-49,Associate's degree,White,Married,Nothing in particular,Never,Independent,"$75,000 - $100,000",Moderate,More than 4,Home-maker


In [5]:
# Compare demographic marginal shares: full Twin-2K-500 (N=2,058) vs Mega-Study subset (N=1,788)
mega_persona_df = persona_base_df.loc[mega_pids]
comp_rows = []
for col in ["SEX", "AGE", "US_REG", "EDULEVEL", "RACE", "US_PRTY", "US_INC", "PARTY_LR"]:
    s_all = persona_base_df[col].value_counts(normalize=True, dropna=False) * 100
    s_mega = mega_persona_df[col].value_counts(normalize=True, dropna=False) * 100
    for lvl in persona.GLOBAL_PERSONA_LEVELS[col]:
        if lvl in s_all.index or lvl in s_mega.index:
            p_all = float(s_all.get(lvl, 0.0))
            p_mega = float(s_mega.get(lvl, 0.0))
            comp_rows.append({
                "attr": col,
                "level": lvl,
                "Twin-2K-500 (%)": round(p_all, 2),
                "Mega-Study (%)": round(p_mega, 2),
                "diff (pp)": round(p_mega - p_all, 2),
            })
display(pd.DataFrame(comp_rows))

,attr,level,Twin-2K-500 (%),Mega-Study (%),diff (pp)
0,SEX,Male,49.27,48.94,-0.33
1,SEX,Female,50.73,51.06,0.33
2,AGE,18-29,18.85,17.73,-1.12
3,AGE,30-49,35.71,34.84,-0.87
4,AGE,50-64,31.97,33.11,1.14
5,AGE,65+,13.46,14.32,0.86
6,US_REG,Northeast,16.62,16.89,0.27
7,US_REG,Midwest,18.08,18.23,0.16
8,US_REG,South,40.52,40.10,-0.42
9,US_REG,West,24.78,24.78,-0.01


## 3. Manual verification catalog (`question_audit.csv`) and Kev preview helper

In [6]:
from data.base import add_one_hot, make_record, make_view
from models.kev_pointer import DEFAULT_BASE, KevEncoder

audit_path = MEGA_DIR / "question_audit.csv"
audit_raw_df = pd.read_csv(audit_path, keep_default_na=False)

overrides_path = MEGA_DIR / "question_overrides.json"
if overrides_path.exists():
    question_overrides = json.loads(overrides_path.read_text(encoding="utf-8"))
    if not isinstance(question_overrides, dict):
        raise TypeError(f"{overrides_path.name} must contain a JSON object, got {type(question_overrides).__name__}")
else:
    question_overrides = {}

tags_path = MEGA_DIR / "question_tags.json"
if tags_path.exists():
    question_tags = json.loads(tags_path.read_text(encoding="utf-8"))
    if not isinstance(question_tags, dict):
        raise TypeError(f"{tags_path.name} must contain a JSON object, got {type(question_tags).__name__}")
    for rk_tag, tag_list in question_tags.items():
        if not isinstance(tag_list, list) or not all(isinstance(t, str) and t.strip() for t in tag_list):
            raise ValueError(f"{rk_tag}: question_tags entry must be a list of non-empty strings")
else:
    question_tags = {}

ALLOWED_OVERRIDE_FIELDS = {"stem", "subitems", "options"}


def normalize_multiline_override(text: str) -> str:
    lines = [ln.rstrip() for ln in text.replace("\r\n", "\n").replace("\r", "\n").split("\n")]
    return "\n".join(lines).strip()


valid_override_keys = set()
expanded_rows = []
for _, r in audit_raw_df.iterrows():
    idx_str = str(r["catalog_idx"])
    row_key = f"{idx_str}:{r['study']}:{r['QuestionID']}"
    valid_override_keys.add(row_key)
    inc = r["include"] == "True" if isinstance(r["include"], str) else bool(r["include"])
    if ".." in idx_str:
        s_idx, e_idx = map(int, idx_str.split(".."))
        if not r["QuestionID"].startswith("1..200_"):
            raise ValueError(f"Unexpected range QuestionID pattern: {r['QuestionID']!r}")
        base_q = r["QuestionID"].split("_", 1)[1]
        if e_idx - s_idx + 1 != 200:
            raise ValueError(f"Expected 200 items in range {idx_str}")
        for offset, idx in enumerate(range(s_idx, e_idx + 1), 1):
            expanded_rows.append({
                "catalog_idx": idx,
                "override_key": row_key,
                "study": r["study"],
                "QuestionID": f"{offset}_{base_q}",
                "family": r["family"],
                "include": inc,
                "kev_type": r["kev_type"],
                "context_source": r["context_source"],
                "context_sep": r["context_sep"],
                "drop_options": r["drop_options"],
                "tags": "; ".join(question_tags[row_key]) if row_key in question_tags else "",
                "review_note": r["review_note"],
            })
    else:
        expanded_rows.append({
            "catalog_idx": int(idx_str),
            "override_key": row_key,
            "study": r["study"],
            "QuestionID": r["QuestionID"],
            "family": r["family"],
            "include": inc,
            "kev_type": r["kev_type"],
            "context_source": r["context_source"],
            "context_sep": r["context_sep"],
            "drop_options": r["drop_options"],
            "tags": "; ".join(question_tags[row_key]) if row_key in question_tags else "",
            "review_note": r["review_note"],
        })

unknown_override_keys = sorted(set(question_overrides.keys()) - valid_override_keys)
if unknown_override_keys:
    raise KeyError(f"Unknown row keys in {overrides_path.name}: {unknown_override_keys}")
unknown_tag_keys = sorted(set(question_tags.keys()) - valid_override_keys)
if unknown_tag_keys:
    raise KeyError(f"Unknown row keys in {tags_path.name}: {unknown_tag_keys}")

audit_exp_df = pd.DataFrame(expanded_rows).set_index("catalog_idx").sort_index()
if not audit_exp_df.index.equals(catalog_df.index):
    raise ValueError("Expanded question_audit.csv indices do not match catalog_df.index")
if not (audit_exp_df["study"] == catalog_df["study"]).all() or not (audit_exp_df["QuestionID"] == catalog_df["QuestionID"]).all():
    raise ValueError("Expanded question_audit.csv (study, QuestionID) does not match catalog_df")

for col in ("override_key", "family", "include", "kev_type", "context_source", "context_sep", "drop_options", "tags", "review_note"):
    catalog_df[col] = audit_exp_df[col]

# Validate question_overrides.json schema and bounds against catalog_df
for rk, ov in question_overrides.items():
    if not isinstance(ov, dict) or not ov:
        raise ValueError(f"{rk}: override entry must be a non-empty dict")
    extra_fields = set(ov.keys()) - ALLOWED_OVERRIDE_FIELDS
    if extra_fields:
        raise ValueError(f"{rk}: unknown override fields {sorted(extra_fields)}")
    matched_cat = catalog_df[catalog_df["override_key"] == rk]
    if matched_cat.empty:
        raise KeyError(f"{rk}: override key did not match any catalog_df row")
    if ".." in rk.split(":", 1)[0] and "stem" in ov:
        raise ValueError(f"{rk}: static stem override is not allowed on collapsed loop rows")
    cat0 = matched_cat.iloc[0]
    if "stem" in ov:
        if not isinstance(ov["stem"], str):
            raise TypeError(f"{rk}: stem override must be str, got {type(ov['stem']).__name__}")
    if "subitems" in ov:
        if not isinstance(ov["subitems"], dict) or not ov["subitems"]:
            raise ValueError(f"{rk}: subitems override must be a non-empty dict")
        n_rows = len(cat0["Rows"])
        if n_rows == 0:
            raise ValueError(f"{rk}: subitems override set on row without Matrix Rows")
        for sub_k, sub_v in ov["subitems"].items():
            if not isinstance(sub_k, str) or not sub_k.isdigit():
                raise ValueError(f"{rk}: subitem key {sub_k!r} must be a 0-based integer string")
            sub_idx = int(sub_k)
            if not (0 <= sub_idx < n_rows):
                raise IndexError(f"{rk}: subitem index {sub_idx} out of range (0..{n_rows - 1})")
            if not isinstance(sub_v, str) or not normalize_multiline_override(sub_v):
                raise ValueError(f"{rk}: subitem {sub_k!r} override must be a non-empty string")
    if "options" in ov:
        if not isinstance(ov["options"], dict) or not ov["options"]:
            raise ValueError(f"{rk}: options override must be a non-empty dict")
        known_opts = set(cat0["Options_or_Columns"])
        for opt_k, opt_v in ov["options"].items():
            if opt_k not in known_opts:
                raise KeyError(f"{rk}: option override key {opt_k!r} not in {cat0['Options_or_Columns']}")
            if not isinstance(opt_v, str) or not norm_str(opt_v):
                raise ValueError(f"{rk}: option override for {opt_k!r} must be a non-empty string")

# Validate kev_type and context_source syntax across all 1,748 entries
for _, r in catalog_df.iterrows():
    if r["include"]:
        if r["kev_type"] not in ("choice", "score"):
            raise ValueError(f"{r['study']}:{r['QuestionID']} included with invalid kev_type={r['kev_type']!r}")
    else:
        if r["kev_type"] != "":
            raise ValueError(f"{r['study']}:{r['QuestionID']} excluded but has non-empty kev_type={r['kev_type']!r}")
    cs = r["context_source"]
    if cs != "" and cs != "in_block_db" and not cs.startswith("qid:") and not cs.startswith("prefix:"):
        raise ValueError(f"{r['study']}:{r['QuestionID']} invalid context_source={cs!r}")

print(
    f"Verified 1-to-1 alignment between question_audit.csv ({len(audit_raw_df)} rows), "
    f"catalog_df ({len(catalog_df):,} entries), and {overrides_path.name} ({len(question_overrides)} overrides):"
)
display(
    catalog_df.groupby(["family", "include", "kev_type"], as_index=False)
    .agg(
        n_studies=("study", "nunique"),
        n_catalog_entries=("QuestionID", "count"),
        n_subitems=("n_subitems", "sum"),
    )
)

audit_raw_df["tags"] = [
    "; ".join(question_tags[f"{r['catalog_idx']}:{r['study']}:{r['QuestionID']}"])
    if f"{r['catalog_idx']}:{r['study']}:{r['QuestionID']}" in question_tags
    else ""
    for _, r in audit_raw_df.iterrows()
]
print(f"Tagged audit rows in {tags_path.name} ({(audit_raw_df['tags'] != '').sum()} rows):")
display(
    audit_raw_df.loc[
        audit_raw_df["tags"] != "",
        ["catalog_idx", "study", "QuestionID", "family", "include", "kev_type", "context_source", "tags", "review_note"],
    ]
)

Verified 1-to-1 alignment between question_audit.csv (554 rows), catalog_df (1,748 entries), and question_overrides.json (10 overrides):


,family,include,kev_type,n_studies,n_catalog_entries,n_subitems
0,0_CS,False,,1,8,40
1,0_Consent_Check,False,,12,19,19
2,0_DB,False,,18,177,0
3,0_Demographics,False,,1,6,6
4,0_Slider,False,,2,10,10
5,0_TE,False,,11,49,82
6,1_Matrix_Bipolar,True,score,3,12,12
7,2_Matrix_Likert,False,,2,2,12
8,2_Matrix_Likert,True,score,7,29,190
9,3_MC_IdeaEval,True,score,1,1200,1200


Tagged audit rows in question_tags.json (70 rows):


,catalog_idx,study,QuestionID,family,include,kev_type,context_source,tags,review_note
39,39,accuracy_nudges,QID1428,4_MC_Single,False,,,backref,Refers to unspecified set of headlines seen ea...
73,73,affective_priming,QID60,2_Matrix_Likert,True,score,,prior-priming,Administered after randomized open-ended recal...
74,74,affective_priming,QID56,2_Matrix_Likert,False,,,prior-task-dep,Refers directly to respondent's preceding open...
76,76,affective_priming,QID39,2_Matrix_Likert,True,score,in_block_db,prior-priming,Administered after randomized open-ended recal...
80,80,affective_priming,QID43,2_Matrix_Likert,True,score,,prior-priming,Administered after randomized open-ended recal...
...,...,...,...,...,...,...,...,...,...
459,1653,quantitative_intuition,QID31,4_MC_Single,True,choice,,persona,
501,1695,story_beliefs,QID1218314065,4_MC_Single,False,,in_block_db,prior-task-dep,Asks surprise relative to respondent's prior C...
503,1697,story_beliefs,QID242,4_MC_Single,False,,in_block_db,prior-task-dep,Asks surprise relative to respondent's prior C...
529,1723,story_beliefs,QID346,4_MC_Single,False,,in_block_db,prior-task-dep,Asks surprise relative to respondent's prior C...


In [7]:
kev_encoder = KevEncoder.for_base(DEFAULT_BASE)

included_audit_lookup = {
    (r["study"], r["QuestionID"]): r
    for _, r in catalog_df[catalog_df["include"]].iterrows()
}

stem_override_lookup = {
    (r["study"], r["QuestionID"]): normalize_multiline_override(question_overrides[r["override_key"]]["stem"])
    for _, r in catalog_df.iterrows()
    if r["override_key"] in question_overrides and "stem" in question_overrides[r["override_key"]]
}


def make_base_slot(study: str, qid: str, rid: str | None = None) -> str:
    slug = study.replace("_", "-")
    m = re.match(r"^(\d+)_(QID.+)$", qid)
    if m:
        loop_idx, base_q = int(m.group(1)), m.group(2)
        loop_fmt = f"{loop_idx:03d}" if study == "idea_evaluation" else str(loop_idx)
        slot = f"{slug}-{base_q}_{loop_fmt}"
    else:
        slot = f"{slug}-{qid}"
    if rid is not None:
        slot = f"{slot}_{rid}"
    return slot


TERMINAL_PUNCT = frozenset(".?!:;…\"')")


def resolve_context(ctx_spec: str, study: str, qid: str, in_block_db_txt: str | None, qid_to_text: dict[str, str]) -> str:
    if ctx_spec == "":
        return ""
    if ctx_spec == "in_block_db":
        if not in_block_db_txt:
            raise ValueError(f"{study}:{qid} requires in_block_db context, but none found in block")
        return in_block_db_txt
    if ctx_spec.startswith("qid:"):
        ref_qids = ctx_spec.removeprefix("qid:").split("+")
        parts = []
        for rq in ref_qids:
            if rq not in qid_to_text or not qid_to_text[rq]:
                raise ValueError(f"{study}:{qid} missing referenced context {rq!r}")
            parts.append(qid_to_text[rq])
        return "\n\n".join(parts)
    if ctx_spec.startswith("prefix:"):
        pfx = ctx_spec.removeprefix("prefix:").strip()
        if not pfx:
            raise ValueError(f"{study}:{qid} empty prefix: context_source")
        return pfx
    raise ValueError(f"{study}:{qid} unsupported context_source={ctx_spec!r}")


def decode_context_sep(sep_spec: str) -> str:
    if sep_spec == "":
        return "\n\n"
    return sep_spec.replace("\\n", "\n").replace("\\t", "\t")


def join_context_and_question(ctx_txt: str, q_txt: str, sep: str) -> str:
    if ctx_txt and q_txt:
        return f"{ctx_txt}{sep}{q_txt}"
    if ctx_txt:
        return ctx_txt
    return q_txt


def join_matrix_stem_and_row(stem: str, row_str: str, *, ctx_txt: str, q_txt: str, sep_spec: str) -> str:
    if not stem:
        return row_str
    if not q_txt and ctx_txt and sep_spec != "":
        return f"{ctx_txt}{decode_context_sep(sep_spec)}{row_str}"
    if stem[-1] in TERMINAL_PUNCT:
        return f"{stem} {row_str}"
    return f"{stem}\n\n{row_str}"


def apply_option_overrides(study: str, qid: str, raw_list: list[str], drop_set: set[str], opt_overrides: dict[str, str] | None) -> tuple[list[str], list[str]]:
    if opt_overrides is not None:
        missing_keys = sorted(set(opt_overrides.keys()) - set(raw_list))
        if missing_keys:
            raise KeyError(f"{study}:{qid} option override keys {missing_keys} not found in raw options {raw_list}")
    kept_raw = [o for o in raw_list if o not in drop_set]
    if opt_overrides is None:
        kept_final = list(kept_raw)
    else:
        kept_final = [norm_str(opt_overrides[o]) if o in opt_overrides else o for o in kept_raw]
    if len(kept_final) < 2 or len(set(kept_final)) != len(kept_final):
        raise ValueError(f"{study}:{qid} invalid kept options after override: {kept_final}")
    return kept_raw, kept_final


def extract_family_long_df(target_family: str) -> pd.DataFrame:
    """Extract respondent-level long-format rows for `target_family` according to `question_audit.csv` and `question_overrides.json`."""
    raw_obs = []
    raw_qtext_variants: dict[tuple[str, str], set[str]] = {}
    target_studies = sorted({s for (s, _), meta in included_audit_lookup.items() if meta["family"] == target_family})
    for study in target_studies:
        df = study_dfs[study]
        for pid_str, raw_json in zip(df["PID"], df["survey_json_with_human_response"], strict=True):
            pid_int = int(pid_str.removeprefix("pid_"))
            obj = json.loads(raw_json)
            blocks = [b for b, _ in iter_blocks(obj["Elements"])]
            qid_to_text = {}
            for b in blocks:
                for q in b["Questions"]:
                    q_id = q["QuestionID"]
                    raw_clean = clean_text(study, q_id, q["QuestionText"])
                    if (study, q_id) in stem_override_lookup:
                        seen_set = raw_qtext_variants.setdefault((study, q_id), set())
                        seen_set.add(raw_clean)
                        if len(seen_set) > 1:
                            raise ValueError(
                                f"{study}:{q_id} has multiple raw QuestionText variants across respondents; "
                                "static stem override is disallowed"
                            )
                        qid_to_text[q_id] = stem_override_lookup[(study, q_id)]
                    else:
                        qid_to_text[q_id] = raw_clean

            for b in blocks:
                bname = b["BlockName"].strip()
                in_block_db_txt = None
                for q in b["Questions"]:
                    qid = q["QuestionID"]
                    qt = q["QuestionType"]
                    q_txt = qid_to_text[qid]
                    if qt == "DB":
                        in_block_db_txt = q_txt
                        continue

                    if (study, qid) not in included_audit_lookup:
                        continue
                    meta = included_audit_lookup[(study, qid)]
                    if meta["family"] != target_family:
                        continue

                    ans = q["Answers"]
                    if not ans:
                        continue

                    ov = question_overrides[meta["override_key"]] if meta["override_key"] in question_overrides else None
                    opt_ov = ov["options"] if ov is not None and "options" in ov else None
                    sub_ov = ov["subitems"] if ov is not None and "subitems" in ov else None

                    kev_type = meta["kev_type"]
                    drop_set = {x.strip() for x in meta["drop_options"].split("|") if x.strip()}
                    ctx_txt = resolve_context(meta["context_source"], study, qid, in_block_db_txt, qid_to_text)
                    sep_spec = meta["context_sep"]
                    stem = join_context_and_question(ctx_txt, q_txt, decode_context_sep(sep_spec))
                    if not stem:
                        raise ValueError(f"{study}:{qid} empty question stem")

                    if qt == "MC":
                        pos_raw = ans["SelectedByPosition"]
                        if pos_raw is None:
                            continue
                        pos = int(pos_raw)
                        raw_opts = [norm_str(o) for o in q["Options"]]
                        sel_raw = norm_str(ans["SelectedText"])
                        if raw_opts[pos - 1] != sel_raw:
                            raise ValueError(f"{study}:{qid} SelectedByPosition/SelectedText mismatch")
                        if sel_raw in drop_set:
                            continue
                        kept_raw_opts, kept_opts = apply_option_overrides(study, qid, raw_opts, drop_set, opt_ov)
                        ans_idx = kept_raw_opts.index(sel_raw)
                        sel_final = kept_opts[ans_idx]
                        raw_obs.append({
                            "pid": pid_int,
                            "base_slot": make_base_slot(study, qid),
                            "label_prefix": f"{study} ({bname})",
                            "label_suffix": qid,
                            "question": stem,
                            "type": kev_type,
                            "option_texts": tuple(kept_opts),
                            "answer_code": ans_idx + 1,
                            "answer_text": sel_final,
                        })
                    elif qt == "Matrix":
                        sel = q["Settings"]["Selector"]
                        raw_cols = [norm_str(c) for c in q["Columns"]]
                        raw_rows = [norm_str(r_str) for r_str in q["Rows"]]
                        if sub_ov is None:
                            rows = raw_rows
                        else:
                            if max(int(k) for k in sub_ov.keys()) >= len(raw_rows):
                                raise IndexError(f"{study}:{qid} subitems override index exceeds respondent Rows length {len(raw_rows)}")
                            rows = [
                                normalize_multiline_override(sub_ov[str(i)]) if str(i) in sub_ov else r_str
                                for i, r_str in enumerate(raw_rows)
                            ]
                        rows_id = [str(x) for x in q["RowsID"]] if "RowsID" in q else [str(i + 1) for i in range(len(rows))]
                        pos_list = ans["SelectedByPosition"]
                        txt_list = ans["SelectedText"]
                        if not (len(rows) == len(rows_id) == len(pos_list) == len(txt_list)):
                            raise ValueError(f"{study}:{qid} Matrix length mismatch")
                        kept_raw_cols, kept_opts = apply_option_overrides(study, qid, raw_cols, drop_set, opt_ov)

                        if sel == "Bipolar":
                            if len(rows) != 1:
                                raise ValueError(f"{study}:{qid} expected 1 row in Matrix:Bipolar, got {len(rows)}")
                            if pos_list[0] is None:
                                continue
                            pos = int(pos_list[0])
                            sel_raw = norm_str(txt_list[0])
                            if raw_cols[pos - 1] != sel_raw:
                                raise ValueError(f"{study}:{qid} Bipolar SelectedText mismatch")
                            if sel_raw in drop_set:
                                continue
                            ans_idx = kept_raw_cols.index(sel_raw)
                            sel_final = kept_opts[ans_idx]
                            raw_obs.append({
                                "pid": pid_int,
                                "base_slot": make_base_slot(study, qid),
                                "label_prefix": f"{study} ({bname})",
                                "label_suffix": qid,
                                "question": stem,
                                "type": kev_type,
                                "option_texts": tuple(kept_opts),
                                "answer_code": ans_idx + 1,
                                "answer_text": sel_final,
                            })
                        elif sel == "Likert":
                            for rid, row_str, p_val, t_val in zip(rows_id, rows, pos_list, txt_list, strict=True):
                                if p_val is None:
                                    continue
                                pos = int(p_val)
                                sel_raw = norm_str(t_val)
                                if raw_cols[pos - 1] != sel_raw:
                                    raise ValueError(f"{study}:{qid}_{rid} Likert SelectedText mismatch")
                                if sel_raw in drop_set:
                                    continue
                                ans_idx = kept_raw_cols.index(sel_raw)
                                sel_final = kept_opts[ans_idx]
                                raw_obs.append({
                                    "pid": pid_int,
                                    "base_slot": make_base_slot(study, qid, rid),
                                    "label_prefix": f"{study} ({bname})",
                                    "label_suffix": row_str,
                                    "question": join_matrix_stem_and_row(stem, row_str, ctx_txt=ctx_txt, q_txt=q_txt, sep_spec=sep_spec),
                                    "type": kev_type,
                                    "option_texts": tuple(kept_opts),
                                    "answer_code": ans_idx + 1,
                                    "answer_text": sel_final,
                                })
                        else:
                            raise ValueError(f"Unsupported Matrix selector: {sel!r}")
                    else:
                        raise ValueError(f"Unsupported QuestionType in included set: {qt!r}")

    obs_df = pd.DataFrame(raw_obs)
    v_map = {}
    for base_slot, grp in obs_df.groupby("base_slot", sort=False):
        variants = sorted(set(zip(grp["question"], grp["option_texts"], strict=True)))
        if len(variants) == 1:
            v_map[(base_slot, variants[0][0], variants[0][1])] = base_slot
        else:
            for v_idx, (q_str, opt_tup) in enumerate(variants, 1):
                v_map[(base_slot, q_str, opt_tup)] = f"{base_slot}_v{v_idx:03d}"

    obs_df["v"] = [
        v_map[(bs, q_str, opt_tup)]
        for bs, q_str, opt_tup in zip(obs_df["base_slot"], obs_df["question"], obs_df["option_texts"], strict=True)
    ]
    obs_df = obs_df.sort_values(["v", "pid"], ignore_index=True)
    if obs_df.duplicated(subset=["v", "pid"]).any():
        raise ValueError(f"Duplicate (v, pid) in {target_family}")

    out_df = persona_base_df.loc[obs_df["pid"]].reset_index(drop=True)
    out_df["v"] = obs_df["v"]
    out_df["label"] = obs_df["label_prefix"] + ": " + obs_df["label_suffix"]
    out_df["question"] = obs_df["question"]
    out_df["type"] = obs_df["type"]
    out_df["reversed"] = False
    out_df["option_codes"] = [list(range(1, len(opts) + 1)) for opts in obs_df["option_texts"]]
    out_df["option_texts"] = [list(opts) for opts in obs_df["option_texts"]]
    out_df["answer_code"] = obs_df["answer_code"].astype("int64")
    out_df["answer_text"] = obs_df["answer_text"]
    return out_df


def preview_long_question(long_df: pd.DataFrame, v_id: str, attrs=None, case_id: str | None = None, width: int = 95):
    sub_q = long_df[long_df["v"] == v_id]
    if sub_q.empty:
        raise KeyError(f"v={v_id!r} not found in long_df")
    ref_row = sub_q.iloc[0] if case_id is None else sub_q[sub_q["CASEID"] == case_id].iloc[0]
    q_type = ref_row["type"]
    opt_codes = ref_row["option_codes"]
    opt_texts = ref_row["option_texts"]

    if attrs is None:
        cols = [c for c in PERSONA_COLS if c != "c_alphan" and pd.notna(ref_row[c])]
        probs = [1.0 if c == ref_row["answer_code"] else 0.0 for c in opt_codes]
        header_tag = f"CASEID={ref_row['CASEID']!r} (question n={len(sub_q)})"
    else:
        cols = list(attrs)
        mask = pd.Series(True, index=sub_q.index)
        for c in cols:
            mask &= sub_q[c] == ref_row[c]
        sub_grp = sub_q.loc[mask]
        counts = sub_grp["answer_code"].value_counts().reindex(opt_codes, fill_value=0)
        probs = (counts / counts.sum()).astype(float).tolist()
        header_tag = f"subpop n={len(sub_grp)}"

    state_lines = [f"{persona.PERSONA_NAMES['c_alphan']}: {ref_row['c_alphan']}"]
    state_lines.extend(f"{persona.PERSONA_NAMES[c]}: {ref_row[c]}" for c in cols)
    if q_type == "score":
        criteria = list(opt_texts)
        target = {str(i): round(p, 4) for i, p in enumerate(probs)}
    elif q_type == "choice":
        criteria = {t: None for t in opt_texts}
        target = {t: round(p, 4) for t, p in zip(opt_texts, probs)}
    else:
        raise ValueError(f"Unexpected type {q_type!r}")

    view = {
        "state": "\n".join(state_lines),
        "questions": {
            f"twin2k-mega:{v_id}": {
                "type": q_type,
                "instructions": ref_row["question"],
                "criteria": criteria,
                "target": target,
            }
        },
    }
    header, rendered = kev_encoder.preview(view)
    wrapped = "\n".join(
        line if line.startswith("<|") else textwrap.fill(line, width=width)
        for line in rendered.splitlines()
    )
    print(f"\n=== {v_id} (type={q_type!r}, {header_tag}, {header}) ===")
    print(wrapped)
    print("Target distribution:", json.dumps(target))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

## 4. Substantive question families in Kev format

In [8]:
# 4.1 Family 1: Matrix — Bipolar (12 slots across accuracy_nudges, infotainment, promiscuous_donors)
# Note: Unlike Twin-2K-500 binary 'opt1:opt2' MPLs, all 12 Mega-Study Matrix:Bipolar items are single-row
# 6-point or 7-point ordinal bipolar rating scales whose scale labels live in Columns (kev_type='score').
bipolar_long_df = extract_family_long_df("1_Matrix_Bipolar")
bipolar_counts = bipolar_long_df.groupby("v")["CASEID"].count()
print(
    f"bipolar_long_df: {len(bipolar_long_df):,} rows x {bipolar_long_df.shape[1]} cols | "
    f"{bipolar_long_df['CASEID'].nunique():,} respondents | "
    f"{bipolar_long_df['v'].nunique():,} unique questions (n/question: min={bipolar_counts.min()}, "
    f"median={bipolar_counts.median():.0f}, max={bipolar_counts.max()})"
)
display(bipolar_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["accuracy-nudges-QID1581", "promiscuous-donors-QID478_v001"]:
    preview_long_question(bipolar_long_df, v_id)

bipolar_long_df: 9,293 rows x 24 cols | 1,246 respondents | 1,019 unique questions (n/question: min=1, median=8, max=1003)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_texts,answer_code,answer_text
0,twin_1,Female,30-49,High school or some college,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,score,"[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",1,1 - Verifiably correct
1,twin_2,Female,50-64,High school or some college,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,score,"[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",1,1 - Verifiably correct
2,twin_3,Male,65+,High school or some college,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,score,"[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",2,2
3,twin_4,Male,50-64,College graduate/some postgrad,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,score,"[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",4,4



=== accuracy-nudges-QID1581 (type='score', CASEID='twin_1' (question n=1003), 156 tokens; state 108 tokens (no truncation, strict=True)) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
When you share information with others on social media, is it more important that the
information is:
<|box_start|>1 - Verifiably correct<|box_end|>
<|box_start|>2<|box_end|>
<|box_start|>3<|box_end|>
<|box_start|>4<|box_end|>
<|box_start|>5<|box_end|>
<|box_start|>6 - Entertaining<|box_end|>
<|fim_suffix|>
Target distribution: {"0": 1.0, "1": 0.0, "2": 0.0, "3": 0.0, "4": 0.0, "5": 0.0}

=== promiscuous-donors-QID478_

In [9]:
# 4.2 Family 2: Matrix — Likert (29 included batteries across 8 studies -> 190 unique questions, all 'score')
likert_long_df = extract_family_long_df("2_Matrix_Likert")
likert_counts = likert_long_df.groupby("v")["CASEID"].count()
print(
    f"likert_long_df: {len(likert_long_df):,} rows x {likert_long_df.shape[1]} cols | "
    f"{likert_long_df['CASEID'].nunique():,} respondents | "
    f"{likert_long_df['v'].nunique():,} unique questions (n/question: min={likert_counts.min()}, "
    f"median={likert_counts.median():.0f}, max={likert_counts.max()})"
)
display(likert_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["privacy-QID11_1", "quantitative-intuition-QID1_1", "hiring-algorithms-QID22_1"]:
    preview_long_question(likert_long_df, v_id)

likert_long_df: 134,948 rows x 24 cols | 1,734 respondents | 190 unique questions (n/question: min=180, median=502, max=1435)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_texts,answer_code,answer_text
0,twin_5,Female,30-49,Associate's degree,affective-priming-QID39_1,affective_priming (DV2 Predictability): I woul...,score,"[Completely disagree 1, 2, 3, 4, 5, 6, Complet...",3,3
1,twin_6,Male,50-64,High school or some college,affective-priming-QID39_1,affective_priming (DV2 Predictability): I woul...,score,"[Completely disagree 1, 2, 3, 4, 5, 6, Complet...",3,3
2,twin_10,Male,30-49,High school or some college,affective-priming-QID39_1,affective_priming (DV2 Predictability): I woul...,score,"[Completely disagree 1, 2, 3, 4, 5, 6, Complet...",2,2
3,twin_15,Male,50-64,Associate's degree,affective-priming-QID39_1,affective_priming (DV2 Predictability): I woul...,score,"[Completely disagree 1, 2, 3, 4, 5, 6, Complet...",2,2



=== privacy-QID11_1 (type='score', CASEID='twin_21' (question n=187), 268 tokens; state 98 tokens (no truncation, strict=True)) ===
<|fim_prefix|>
Country: United States
Region: South
Sex: Female
Age: 50-64
Education: College graduate/some postgrad
Race: White
Marital status: Married
Religion: Catholic
Attendance of religious services: Never
Political party: Republican
Income: $100,000 or more
Political ideology: Moderate
How many people, including yourself, live in your household?: 2
Main activity: Full-time employment
<|fim_middle|>
Please read the description of how online advertising could work in the future below and answer
the following question: Your online activity on your desktop computer, laptop, and mobile
devices will not be tracked, and your data will not leave your devices. This means that you
personally or your device will not be identifiable on the Internet. You will receive
advertising targeted only to the context of the website you are visiting. For instance, if you


In [10]:
# 4.3 Family 3: MC — Idea Evaluation Loop-and-Merge (6 arms x 200 app ideas = 1,200 unique questions, all 'score')
idea_long_df = extract_family_long_df("3_MC_IdeaEval")
idea_counts = idea_long_df.groupby("v")["CASEID"].count()
print(
    f"idea_long_df: {len(idea_long_df):,} rows x {idea_long_df.shape[1]} cols | "
    f"{idea_long_df['CASEID'].nunique():,} respondents | "
    f"{idea_long_df['v'].nunique():,} unique questions (n/question: min={idea_counts.min()}, "
    f"median={idea_counts.median():.0f}, max={idea_counts.max()})"
)
display(idea_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["idea-evaluation-QID7_001", "idea-evaluation-QID14_001"]:
    preview_long_question(idea_long_df, v_id)

idea_long_df: 23,480 rows x 24 cols | 1,174 respondents | 1,200 unique questions (n/question: min=8, median=19, max=33)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_texts,answer_code,answer_text
0,twin_124,Male,30-49,College graduate/some postgrad,idea-evaluation-QID13_001,idea_evaluation (idea evaluation - human ideas...,score,"[Very uncreative, Somewhat uncreative, Neither...",2,Somewhat uncreative
1,twin_144,Female,50-64,College graduate/some postgrad,idea-evaluation-QID13_001,idea_evaluation (idea evaluation - human ideas...,score,"[Very uncreative, Somewhat uncreative, Neither...",2,Somewhat uncreative
2,twin_157,Female,50-64,Associate's degree,idea-evaluation-QID13_001,idea_evaluation (idea evaluation - human ideas...,score,"[Very uncreative, Somewhat uncreative, Neither...",5,Very creative
3,twin_236,Female,30-49,College graduate/some postgrad,idea-evaluation-QID13_001,idea_evaluation (idea evaluation - human ideas...,score,"[Very uncreative, Somewhat uncreative, Neither...",3,Neither uncreative nor creative



=== idea-evaluation-QID7_001 (type='score', CASEID='twin_22' (question n=24), 281 tokens; state 98 tokens (no truncation, strict=True)) ===
<|fim_prefix|>
Country: United States
Region: South
Sex: Female
Age: 30-49
Education: College graduate/some postgrad
Race: Other
Marital status: Married
Religion: Agnostic
Attendance of religious services: Never
Political party: Democrat
Income: $100,000 or more
Political ideology: Liberal
How many people, including yourself, live in your household?: 3
Main activity: Self-employed
<|fim_middle|>
HOW COULD SMARTPHONES HELP THEIR USERS BE HEALTHIER? We are interested in new ideas for
smartphone apps that will help their users keep a healthier lifestyle. We will show you ideas
for smartphone apps priced at $0.99 that would help their users keep a healthier lifestyle. You
will be shown 20 ideas , one after the other. Your task is to judge how creative each idea is.
Below is an idea: Idea: It is an app for eating the right thing. It listens in the back

In [11]:
# 4.4 Family 4: MC — Single-Select (208 included slots across 15 studies -> 458 unique questions: 106 'choice' + 352 'score')
mc_long_df = extract_family_long_df("4_MC_Single")
mc_counts = mc_long_df.groupby("v")["CASEID"].count()
print(
    f"mc_long_df: {len(mc_long_df):,} rows x {mc_long_df.shape[1]} cols | "
    f"{mc_long_df['CASEID'].nunique():,} respondents | "
    f"{mc_long_df['v'].nunique():,} unique questions ("
    f"{mc_long_df[mc_long_df['type'] == 'choice']['v'].nunique()} 'choice', "
    f"{mc_long_df[mc_long_df['type'] == 'score']['v'].nunique()} 'score'; "
    f"n/question: min={mc_counts.min()}, median={mc_counts.median():.0f}, max={mc_counts.max()})"
)
display(mc_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["context-effects-QID544", "accuracy-nudges-QID1200", "junk-fees-QID10_v001", "preference-redistribution-QID3", "story-beliefs-QID332_v001"]:
    preview_long_question(mc_long_df, v_id)

mc_long_df: 119,312 rows x 24 cols | 1,763 respondents | 426 unique questions (112 'choice', 314 'score'; n/question: min=6, median=107, max=1435)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_texts,answer_code,answer_text
0,twin_1,Female,30-49,High school or some college,accuracy-nudges-QID1200,accuracy_nudges (Fake1_SM): QID1200,score,"[Extremely unlikely, Moderately unlikely, Slig...",5,Moderately likely
1,twin_2,Female,50-64,High school or some college,accuracy-nudges-QID1200,accuracy_nudges (Fake1_SM): QID1200,score,"[Extremely unlikely, Moderately unlikely, Slig...",1,Extremely unlikely
2,twin_3,Male,65+,High school or some college,accuracy-nudges-QID1200,accuracy_nudges (Fake1_SM): QID1200,score,"[Extremely unlikely, Moderately unlikely, Slig...",1,Extremely unlikely
3,twin_4,Male,50-64,College graduate/some postgrad,accuracy-nudges-QID1200,accuracy_nudges (Fake1_SM): QID1200,score,"[Extremely unlikely, Moderately unlikely, Slig...",1,Extremely unlikely



=== context-effects-QID544 (type='choice', CASEID='twin_1' (question n=501), 332 tokens; state 108 tokens (no truncation, strict=True)) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
Practice Scenario: Office Chair
The office chair you have now is so old that it is starting to break. Unfortunately, the same
model is no longer available anywhere, so you have to look for something else. On the website
of a furniture store, you will come across different models in different price categories. Of
course, you want an office chair that matches the rest of your furniture. After you’ve looked
at which office

In [12]:
# 4.5 Audit of excluded families and excluded substantive items
excluded_df = audit_raw_df[audit_raw_df["include"].astype(str) == "False"]
print(f"Excluded audit rows: {len(excluded_df)} (covering {(~catalog_df['include']).sum()} catalog entries)")
display(
    catalog_df[~catalog_df["include"]]
    .groupby(["family", "QuestionType", "Selector"], as_index=False)
    .agg(n_studies=("study", "nunique"), n_slots=("QuestionID", "count"), n_subitems=("n_subitems", "sum"))
)
print("Excluded items within 2_Matrix_Likert and 4_MC_Single:")
display(
    audit_raw_df[
        (audit_raw_df["include"].astype(str) == "False")
        & audit_raw_df["family"].isin(["2_Matrix_Likert", "4_MC_Single"])
    ][["catalog_idx", "study", "QuestionID", "family", "tags", "review_note", "QuestionText"]]
)

Excluded audit rows: 301 (covering 301 catalog entries)


,family,QuestionType,Selector,n_studies,n_slots,n_subitems
0,0_CS,CS,VRTL,1,8,40
1,0_Consent_Check,MC,SAHR,2,2,2
2,0_Consent_Check,MC,SAVR,11,17,17
3,0_DB,DB,,18,177,0
4,0_Demographics,MC,SAHR,1,1,1
5,0_Demographics,MC,SAVR,1,5,5
6,0_Slider,Slider,HSLIDER,2,10,10
7,0_TE,TE,ESTB,2,5,5
8,0_TE,TE,FORM,1,7,40
9,0_TE,TE,ML,5,23,23


Excluded items within 2_Matrix_Likert and 4_MC_Single:


,catalog_idx,study,QuestionID,family,tags,review_note,QuestionText
39,39,accuracy_nudges,QID1428,4_MC_Single,backref,Refers to unspecified set of headlines seen ea...,Would you ever consider sharing news like you ...
74,74,affective_priming,QID56,2_Matrix_Likert,prior-task-dep,Refers directly to respondent's preceding open...,Please indicate the extent to which you agree ...
85,85,consumer_minimalism,QID20,4_MC_Single,img-ref,References missing wardrobe images ('pictures ...,"Below, are pictures of two wardrobes. Which wa..."
87,87,consumer_minimalism,QID18,4_MC_Single,img-ref,References missing room images ('pictures of r...,"Below, are pictures of rooms in two different ..."
120,120,digital_certification,QID121,2_Matrix_Likert,prior-task-dep,Refers to whichever sweater respondent selecte...,Please answer the following questions. To what...
124,124,digital_certification,QID167,4_MC_Single,prior-task-dep,Asks how difficult it was to imagine the prior...,How difficult was it to imagine the sweater sc...
219,1413,infotainment,QID1428,4_MC_Single,backref,Refers to unspecified set of headlines seen ea...,Would you ever consider sharing news like you ...
318,1512,junk_fees,QID85,4_MC_Single,backref,Refers to whichever randomized fee blocks resp...,"To what extent do you perceive fees, like the ..."
501,1695,story_beliefs,QID1218314065,4_MC_Single,prior-task-dep,Asks surprise relative to respondent's prior C...,How not surprised or surprised are you after r...
503,1697,story_beliefs,QID242,4_MC_Single,prior-task-dep,Asks surprise relative to respondent's prior C...,How not surprised or surprised are you after r...


## 5. Strict Kev verification and long-format Parquet export

In [13]:
# Combine all 4 included families, verify strict KevEncoder compatibility on every unique question, and export Parquet
non_discrete_included = catalog_df[
    catalog_df["include"]
    & (~catalog_df["family"].isin(["1_Matrix_Bipolar", "2_Matrix_Likert", "3_MC_IdeaEval", "4_MC_Single"]))
]
if not non_discrete_included.empty:
    raise ValueError(f"Unexpected included rows outside discrete families: {non_discrete_included[['study', 'QuestionID', 'family']]}")

twin2k_megastudy_long_df = pd.concat(
    [bipolar_long_df, likert_long_df, idea_long_df, mc_long_df],
    ignore_index=True,
)

expected_long_cols = (
    ["CASEID"]
    + PERSONA_COLS
    + ["v", "label", "question", "type", "reversed", "option_codes", "option_texts", "answer_code", "answer_text"]
)
if list(twin2k_megastudy_long_df.columns) != expected_long_cols:
    raise ValueError(f"Column mismatch: {list(twin2k_megastudy_long_df.columns)} != {expected_long_cols}")

# Verify make_record + make_view + KevEncoder.prepare(strict=True) on every unique v
verify_df = twin2k_megastudy_long_df.copy()
verify_df["year"] = "twin2k-mega"
verify_df["qid"] = "twin2k-mega:" + verify_df["v"]
add_one_hot(verify_df)

max_tokens = 0
max_qid = None
for qid, sub in verify_df.groupby("qid", sort=False):
    rec = make_record(sub.iloc[:1], PERSONA_COLS)
    view = make_view(rec)
    enc_out, _ = kev_encoder.prepare(view)
    if len(enc_out["ids"]) > max_tokens:
        max_tokens = len(enc_out["ids"])
        max_qid = qid

print(
    f"Verified make_record + KevEncoder.prepare(strict=True) across all {verify_df['qid'].nunique():,} unique questions "
    f"(max encoded sequence length = {max_tokens} tokens on {max_qid})."
)

out_parquet = MEGA_DIR / "twin2k_megastudy_US_long.parquet"
twin2k_megastudy_long_df.to_parquet(out_parquet, index=False)

# Reload round-trip check
reloaded_df = pd.read_parquet(out_parquet)
if reloaded_df.shape != twin2k_megastudy_long_df.shape or list(reloaded_df.columns) != expected_long_cols:
    raise ValueError("Parquet round-trip shape/column mismatch")

print(
    f"Saved {out_parquet.name} ({len(twin2k_megastudy_long_df):,} rows x {twin2k_megastudy_long_df.shape[1]} cols | "
    f"{twin2k_megastudy_long_df['CASEID'].nunique():,} respondents | "
    f"{twin2k_megastudy_long_df['v'].nunique():,} unique questions across "
    f"{twin2k_megastudy_long_df['v'].str.split('_').str[0].nunique()} base QID families: "
    f"{twin2k_megastudy_long_df[twin2k_megastudy_long_df['type'] == 'choice']['v'].nunique():,} 'choice', "
    f"{twin2k_megastudy_long_df[twin2k_megastudy_long_df['type'] == 'score']['v'].nunique():,} 'score')"
)

display(
    pd.DataFrame({
        "family": ["1_Matrix_Bipolar", "2_Matrix_Likert", "3_MC_IdeaEval", "4_MC_Single", "Total"],
        "n_base_families": [
            bipolar_long_df["v"].str.split("_").str[0].nunique(),
            likert_long_df["v"].str.split("_").str[0].nunique(),
            idea_long_df["v"].str.split("_").str[0].nunique(),
            mc_long_df["v"].str.split("_").str[0].nunique(),
            twin2k_megastudy_long_df["v"].str.split("_").str[0].nunique(),
        ],
        "n_unique_v": [
            bipolar_long_df["v"].nunique(),
            likert_long_df["v"].nunique(),
            idea_long_df["v"].nunique(),
            mc_long_df["v"].nunique(),
            twin2k_megastudy_long_df["v"].nunique(),
        ],
        "n_rows": [
            len(bipolar_long_df),
            len(likert_long_df),
            len(idea_long_df),
            len(mc_long_df),
            len(twin2k_megastudy_long_df),
        ],
    })
)

study_tag = twin2k_megastudy_long_df["v"].str.extract(r"^([a-z-]+)-QID")[0].str.replace("-", "_")
twin2k_megastudy_long_df_tmp = twin2k_megastudy_long_df.assign(study=study_tag)
display(
    twin2k_megastudy_long_df_tmp.groupby("study", as_index=False)
    .agg(
        n_respondents=("CASEID", "nunique"),
        n_base_families=("v", lambda s: s.str.split("_").str[0].nunique()),
        n_unique_v=("v", "nunique"),
        n_choice_v=("v", lambda s: twin2k_megastudy_long_df_tmp.loc[s.index].loc[twin2k_megastudy_long_df_tmp.loc[s.index, "type"] == "choice", "v"].nunique()),
        n_score_v=("v", lambda s: twin2k_megastudy_long_df_tmp.loc[s.index].loc[twin2k_megastudy_long_df_tmp.loc[s.index, "type"] == "score", "v"].nunique()),
        n_rows=("CASEID", "count"),
    )
)

Verified make_record + KevEncoder.prepare(strict=True) across all 2,835 unique questions (max encoded sequence length = 689 tokens on twin2k-mega:idea-evaluation-QID13_196).


Saved twin2k_megastudy_US_long.parquet (287,033 rows x 24 cols | 1,788 respondents | 2,835 unique questions across 252 base QID families: 112 'choice', 2,723 'score')


,family,n_base_families,n_unique_v,n_rows
0,1_Matrix_Bipolar,12,1019,9293
1,2_Matrix_Likert,29,190,134948
2,3_MC_IdeaEval,6,1200,23480
3,4_MC_Single,205,426,119312
4,Total,252,2835,287033


,study,n_respondents,n_base_families,n_unique_v,n_choice_v,n_score_v,n_rows
0,accuracy_nudges,1003,26,26,0,26,18552
1,affective_priming,1000,5,22,0,22,11000
2,consumer_minimalism,200,3,14,0,14,2800
3,context_effects,501,10,10,10,0,2004
4,default_eric,600,4,4,4,0,1200
5,digital_certification,600,3,3,2,1,1800
6,hiring_algorithms,999,25,81,6,75,48951
7,idea_evaluation,1174,6,1200,0,1200,23480
8,infotainment,300,21,29,0,29,8700
9,junk_fees,400,71,179,57,122,19216


In [14]:
twin2k_megastudy_long_df

,CASEID,c_alphan,US_REG,SEX,AGE,EDULEVEL,RACE,MARITAL,RELIGGRP,ATTEND,...,MAINSTAT,v,label,question,type,reversed,option_codes,option_texts,answer_code,answer_text
0,twin_1,United States,Midwest,Female,30-49,High school or some college,Black,Married,Nothing in particular,A few times a year,...,Part-time employment,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,When you share information with others on soci...,score,False,"[1, 2, 3, 4, 5, 6]","[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",1,1 - Verifiably correct
1,twin_2,United States,Midwest,Female,50-64,High school or some college,White,Divorced,Nothing in particular,Never,...,Retired,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,When you share information with others on soci...,score,False,"[1, 2, 3, 4, 5, 6]","[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",1,1 - Verifiably correct
2,twin_3,United States,South,Male,65+,High school or some college,White,Married,Protestant,Seldom,...,Retired,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,When you share information with others on soci...,score,False,"[1, 2, 3, 4, 5, 6]","[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",2,2
3,twin_4,United States,South,Male,50-64,College graduate/some postgrad,White,Married,Other,Never,...,Full-time employment,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,When you share information with others on soci...,score,False,"[1, 2, 3, 4, 5, 6]","[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",4,4
4,twin_5,United States,South,Female,30-49,Associate's degree,White,Married,Nothing in particular,Never,...,Home-maker,accuracy-nudges-QID1581,accuracy_nudges (Social Media): QID1581,When you share information with others on soci...,score,False,"[1, 2, 3, 4, 5, 6]","[1 - Verifiably correct, 2, 3, 4, 5, 6 - Enter...",3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287028,twin_1861,United States,South,Male,18-29,College graduate/some postgrad,Hispanic,Never been married,Agnostic,Seldom,...,Student,targeting-fairness-QID1396_v002,targeting_fairness (Snack Scenario): QID1396,A snack foods company has developed a new line...,score,False,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1: Not at all fair, 2, 3, 4, 5, 6, 7, 8, 9: V...",6,6
287029,twin_1875,United States,South,Female,18-29,College graduate/some postgrad,White,Never been married,Other,Seldom,...,Full-time employment,targeting-fairness-QID1396_v002,targeting_fairness (Snack Scenario): QID1396,A snack foods company has developed a new line...,score,False,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1: Not at all fair, 2, 3, 4, 5, 6, 7, 8, 9: V...",9,9: Very fair
287030,twin_1929,United States,South,Female,65+,College graduate/some postgrad,White,Married,Catholic,Once a week,...,Self-employed,targeting-fairness-QID1396_v002,targeting_fairness (Snack Scenario): QID1396,A snack foods company has developed a new line...,score,False,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1: Not at all fair, 2, 3, 4, 5, 6, 7, 8, 9: V...",8,8
287031,twin_2019,United States,South,Female,50-64,High school or some college,White,Living with a partner,Agnostic,Never,...,Full-time employment,targeting-fairness-QID1396_v002,targeting_fairness (Snack Scenario): QID1396,A snack foods company has developed a new line...,score,False,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1: Not at all fair, 2, 3, 4, 5, 6, 7, 8, 9: V...",7,7
